# **Latin Morphological Analyser** - Evaluation

This notebook loads the trained model and runs detailed per-feature evaluation on the test set, including a single-sentence prediction example and a full classification report for every morphological feature.

## **Set Up Environment**

In [1]:
import os

### Install Latin BERT

Download the pre-trained Latin BERT model from the GitHub repository and define the path for the model to be used for fine-tuning.

In [2]:
# clone latin bert repo
!git clone https://github.com/dbamman/latin-bert.git
%cd latin-bert

Cloning into 'latin-bert'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 154 (delta 4), reused 27 (delta 3), pack-reused 122 (from 1)
Receiving objects: 100% (154/154), 6.77 MiB | 15.93 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/latin-bert


In [3]:
# Download pre-trained BERT model for Latin
!./scripts/download.sh

--2026-05-18 10:39:10--  https://drive.usercontent.google.com/download?export=download&id=1Te_14UB-DZ8wYPhHGyDg7LadDTjNzpti&confirm=t&uuid=3c5a37c7-2682-42ee-9000-45f8481ed951
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.68.132, 2404:6800:4003:c00::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.68.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 448020480 (427M) [application/octet-stream]
Saving to: ‘latin_bert.tar’

latin_bert.tar      100%[===================>] 427.27M  35.5MB/s    in 13s     

2026-05-18 10:39:25 (31.8 MB/s) - ‘latin_bert.tar’ saved [448020480/448020480]



In [4]:
!ls -lah models/latin_bert | head

total 428M
drwxrwxr-x 2 1001 1001 4.0K Sep  4  2020 .
drwxr-xr-x 4 root root 4.0K May 18 10:39 ..
-rw-rw-r-- 1 1001 1001  503 Sep  4  2020 config.json
-rw-rw-r-- 1 1001 1001 428M Sep  4  2020 pytorch_model.bin
-rw-rw-r-- 1 1001 1001 217K Sep  4  2020 vocab.txt


In [5]:
%cd ..

/content


In [6]:
MODEL_PATH = os.path.join("latin-bert", "models", "latin_bert")

### Import Dependencies

1. Import all required libraries.
2. Mount Google Drive for storage access.
3. Setup HuggingFace access through token.
4. Define available device to be used.

In [7]:
import sys
import json
import torch
from torch.utils.data import DataLoader
from safetensors.torch import load_file
import transformers
from transformers import AutoTokenizer
import datasets
from datasets import load_from_disk
import sklearn
from sklearn.metrics import classification_report
import matplotlib
import matplotlib.pyplot as plt

In [8]:
%matplotlib inline

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

In [11]:
print(torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

True
Using device: cuda


In [12]:
# Import shared definitions (constants, dataclasses, model classes, data collator)
sys.path.append('/content/drive/MyDrive/FYP')
from morph_constants import FEATURE_ORDER, POS_FEATURE_MASK, ALL_FEATS, IGNORE_INDEX
from morphological_analyser import LatinMorphologicalAnalyserConfig, LatinMorphologicalAnalyser, MultiLabelDataCollator

In [13]:
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

Python: 3.12.13
PyTorch: 2.10.0+cu128
Transformers: 5.0.0
Datasets: 4.0.0
scikit-learn: 1.6.1
Matplotlib: 3.10.0
CUDA: 12.8
GPU: Tesla T4


## **Load Model**

In [14]:
# Load the trained model and tokenizer saved
MODEL_DIR = "/content/drive/MyDrive/FYP/morphological_analyser_multihead"

tokenizer   = AutoTokenizer.from_pretrained(MODEL_DIR)

with open(os.path.join(MODEL_DIR, "config.json")) as f:
    config = json.load(f)

label2id_all = config["label2id_all"]
id2label_all = {
    feat: {int(k): v for k, v in d.items()}
    for feat, d in config["id2label_all"].items()
}
pos_feature_mask = config["pos_feature_mask"]
num_labels_per_feat = config["num_labels_per_feat"]

morph_config = LatinMorphologicalAnalyserConfig(
    bert_model_path    = MODEL_PATH,
    num_labels_per_feat = num_labels_per_feat,
    label2id_all        = label2id_all,
    id2label_all = {feat: {str(k): v for k, v in d.items()} for feat, d in id2label_all.items()}, # convert int keys to str for JSON
    pos_feature_mask    = pos_feature_mask,
    pos_embed_dim      = 64,
    dropout            = 0.1,
)

model_loaded = LatinMorphologicalAnalyser(morph_config).to(device)

# Load manually, bypasses from_pretrained's broken key remapping
state_dict = load_file(
    os.path.join(MODEL_DIR, "model.safetensors"),
    device=device
)
missing, unexpected = model_loaded.load_state_dict(state_dict, strict=False)
model = model_loaded.eval()  # set to inference mode

print("Loaded model from", MODEL_DIR)

You are using a model of type latin_morphological_analyser to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: latin-bert/models/latin_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model from /content/drive/MyDrive/FYP/morphological_analyser_multihead


In [15]:
print(model)

LatinMorphologicalAnalyser(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32900, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [16]:
# Rebuild the POS-feature compatibility mask tensor from the loaded vocabularies.
def build_pos_mask_tensor(label2id_pos, device):
    """
    Build a (num_pos, num_features) bool tensor directly from POS_FEATURE_MASK.
    Any POS code not listed in POS_FEATURE_MASK gets all-False (no features).
    Unknown codes are reported so you can add them explicitly.
    """
    num_pos   = len(label2id_pos)
    num_feats = len(FEATURE_ORDER)
    mask      = torch.zeros(num_pos, num_feats, dtype=torch.bool)

    for pos_code, pos_idx in label2id_pos.items():
        if pos_code in POS_FEATURE_MASK:
            for fi, applicable in enumerate(POS_FEATURE_MASK[pos_code]):
                mask[pos_idx, fi] = applicable
        else:
            # find any unmapped codes immediately rather than silently
            # treating them as all false
            print(f"WARNING: POS code '{pos_code}' not in POS_FEATURE_MASK "
                  f"— all features will be masked off for this tag. "
                  f"Add it explicitly.")

    return mask.to(device)


pos_mask_tensor = build_pos_mask_tensor(label2id_all["pos"], device)

## **Load Test Set**

In [17]:
# Load the tokenized test set saved
PROCESSED_DIR = "/content/drive/MyDrive/FYP/processed"
test_ds = load_from_disk(f"{PROCESSED_DIR}/test_ds")
print(f"{len(test_ds)} test examples")

1778 test examples


In [18]:
# Rebuild the data collator (used by evaluate_all_features and collect_predictions_with_confidence).
data_collator_multi = MultiLabelDataCollator(
    tokenizer  = tokenizer,
    feat_names = ALL_FEATS,
)

## **Evaluation**

In [19]:
def predict_sentence(sentence_words, model, tokenizer, device,
                     id2label_all, pos_mask_tensor):
    """
    Given a list of Latin words, return a list of dicts, one per word,
    with the predicted POS and all applicable morphological features.

    Features inapplicable for the predicted POS are returned as "—".
    """
    model.eval()
    words_lower = [w.lower() for w in sentence_words]

    encoding = tokenizer(
        words_lower,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to(device)

    word_ids = encoding.word_ids(batch_index=0)

    with torch.no_grad():
        output = model(**encoding)

    pos_logits     = output.logits[0]           # (seq_len, num_pos)
    feature_logits = output.feature_logits
    pred_pos_ids   = pos_logits.argmax(dim=-1)  # (seq_len,)

    results, seen = [], set()
    for token_idx, word_id in enumerate(word_ids):
        if word_id is None or word_id in seen:
            continue
        seen.add(word_id)

        pos_idx = pred_pos_ids[token_idx].item()
        pos_str = id2label_all["pos"][pos_idx]
        result  = {"word": sentence_words[word_id], "pos": pos_str}

        for feat_idx, feat in enumerate(FEATURE_ORDER):
            if pos_mask_tensor[pos_idx, feat_idx].item():
                pred_idx      = feature_logits[feat][0, token_idx].argmax().item()
                result[feat]  = id2label_all[feat][pred_idx]
            else:
                result[feat] = "—"

        results.append(result)
    return results

In [20]:
# Example:
words = "in principio erat verbum".split()
for r in predict_sentence(words, model, tokenizer, device,
                           id2label_all, pos_mask_tensor):
    print(r)

{'word': 'in', 'pos': 'R-', 'person': '—', 'number': '—', 'tense': '—', 'mood': '—', 'voice': '—', 'gender': '—', 'case': '—', 'degree': '—'}
{'word': 'principio', 'pos': 'Nb', 'person': '—', 'number': 'Sing', 'tense': '—', 'mood': '—', 'voice': '—', 'gender': 'Neut', 'case': 'Abl', 'degree': '—'}
{'word': 'erat', 'pos': 'V-', 'person': '3', 'number': 'Sing', 'tense': 'Imp', 'mood': 'Ind', 'voice': 'Act', 'gender': '—', 'case': '—', 'degree': '—'}
{'word': 'verbum', 'pos': 'Nb', 'person': '—', 'number': 'Sing', 'tense': '—', 'mood': '—', 'voice': '—', 'gender': 'Masc', 'case': 'Nom', 'degree': '—'}


In [21]:
def evaluate_all_features(model, dataset, tokenizer, device,
                           label2id_all, id2label_all, pos_mask_tensor,
                           batch_size=32):
    """
    Per-feature classification reports plus aggregate POS-only
    and full-tag (all features correct simultaneously) accuracies.
    """


    model.eval()

    # run our own DataLoader since Trainer only returns pos_logits
    feat_true = {feat: [] for feat in ALL_FEATS}
    feat_pred = {feat: [] for feat in ALL_FEATS}

    # full-tag accuracy counters
    tag_total         = 0
    tag_correct       = 0
    pos_correct_count = 0

    loader = DataLoader(
        dataset.with_format("torch"),
        batch_size=batch_size,
        collate_fn=data_collator_multi,
    )

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            output = model(
                input_ids      = batch["input_ids"],
                attention_mask = batch["attention_mask"],
            )

        pos_logits     = output.logits                  # (B, T, num_pos)
        feature_logits = output.feature_logits
        pred_pos       = pos_logits.argmax(dim=-1)      # (B, T)
        true_pos       = batch["labels_pos"]            # (B, T)

        valid = true_pos != IGNORE_INDEX                # (B, T)

        # Collect POS
        feat_true["pos"].extend(true_pos[valid].cpu().tolist())
        feat_pred["pos"].extend(pred_pos[valid].cpu().tolist())

        # start per-token tag correctness with POS match
        token_correct = (pred_pos == true_pos)          # (B, T) bool

        # Collect other features
        for feat_idx, feat in enumerate(FEATURE_ORDER):
            true_feat = batch[f"labels_{feat}"]
            pred_feat = feature_logits[feat].argmax(dim=-1)   # compute unconditionally now

            pos_clamp = true_pos.clone()
            pos_clamp[~valid] = 0
            applicable = pos_mask_tensor[pos_clamp, feat_idx] & valid

            if applicable.any():
                feat_true[feat].extend(true_feat[applicable].cpu().tolist())
                feat_pred[feat].extend(pred_feat[applicable].cpu().tolist())

            # applicable -> values must match; inapplicable -> trivially correct
            feat_match    = (pred_feat == true_feat) | (~applicable)
            token_correct = token_correct & feat_match

        # accumulate over valid tokens only
        tag_total         += valid.sum().item()
        tag_correct       += (token_correct & valid).sum().item()
        pos_correct_count += ((pred_pos == true_pos) & valid).sum().item()

    # Per-feature reports
    for feat in ALL_FEATS:
        if not feat_true[feat]:
            continue
        num_labels = len(id2label_all[feat])
        print(f"\n{'='*50}")
        print(f"Feature: {feat.upper()}")
        print(f"{'='*50}")
        print(classification_report(
            feat_true[feat],
            feat_pred[feat],
            labels       = list(range(num_labels)),
            target_names = [id2label_all[feat][i] for i in range(num_labels)],
            zero_division= 0,
        ))

    # aggregate summary
    print(f"\n{'='*50}")
    print("Aggregate accuracies")
    print(f"{'='*50}")
    print(f"POS-only accuracy:  {pos_correct_count / tag_total:.4f}  "
          f"({pos_correct_count}/{tag_total})")
    print(f"Full-tag accuracy:  {tag_correct / tag_total:.4f}  "
          f"({tag_correct}/{tag_total})")

In [22]:
# Run evaluation:
evaluate_all_features(model, test_ds, tokenizer, device,
                      label2id_all, id2label_all, pos_mask_tensor)


Feature: POS
              precision    recall  f1-score   support

          A-       0.69      0.28      0.40       671
          C-       0.97      0.97      0.97      1335
          Df       0.84      0.73      0.78      1472
          Dq       0.91      0.27      0.42        74
          Du       0.71      0.59      0.64        66
          F-       0.00      0.00      0.00         0
          G-       0.90      0.87      0.88       517
          I-       0.91      0.28      0.43        74
          Ma       0.85      0.49      0.62       158
          Mo       0.80      0.23      0.36        35
          Nb       0.60      0.75      0.67      3159
          Ne       0.59      0.36      0.45       629
          Pc       0.00      0.00      0.00         1
          Pd       0.92      0.61      0.73       500
          Pi       0.88      0.84      0.86        90
          Pk       1.00      0.98      0.99        42
          Pp       0.94      0.84      0.89      1101
          Pr 